In [ ]:
# copy from peter use csv instead of other slower storage formats
import os
import uproot
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:

def load_root_file(file_path, branches=None, print_branches=False, print_keys=False):
    all_branches = {}
    with uproot.open(file_path) as file:
        tree = file["trackingNtuple/tree"]
        # Load all ROOT branches into array if not specified
        if branches is None:
            branches = tree.keys()
        # Option to print the branch names
        if print_branches:
            print("Branches:")
            for branch in branches:
                print(f"  {branch}")
        # Option to print the keys
        if print_keys:
            print("Keys:")
            for key in tree.keys():
                print(f"  {key}")
        # Each branch is added to the dictionary
        for branch in branches:
            try:
                all_branches[branch] = (tree[branch].array(library="np"))
            except uproot.KeyInFileError as e:
                print(f"KeyInFileError: {e}")
        # Number of events in file
        all_branches['event'] = tree.num_entries
    return all_branches


In [ ]:
per_particle = [
    'sim_q',
    'sim_pt',
    'sim_pdgId',
    'sim_eta',
    'sim_phi',
    'sim_py', 'sim_px', 'sim_pz',
    'sim_nLay',
    #'simvtx_x', 'simvtx_y', 'simvtx_z',
    "sim_simHitIdx",
    "sim_pca_pt",
    "sim_pca_eta",
    "sim_pca_lambda",
    "sim_pca_cotTheta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]
# loop over these per hit
per_hit = [
    'simhit_x', 'simhit_y', 'simhit_z',
    'simhit_hitType',
    "simhit_isLower",
]
# concatenate the lists
branches_list = per_particle + per_hit 
print(f"branches_list: {branches_list}")

targets = [ # the 5 track parameters we want to predict
    "sim_pca_c",
    "sim_pca_eta",
    "sim_pca_phi",
    "sim_pca_dxy",
    "sim_pca_dz",
]

#file_path = "1000_no_dnn_for_phi.root"
#file_path="/data2/segmentlinking/trackingNtuple_10MuGun.root"
file_path = "/data2/segmentlinking/CMSSW_12_2_0_pre2/trackingNtuple_10mu_pt_0p5_50.root"
#file_path = "/data2/segmentlinking/CMSSW_12_2_0_pre2/trackingNtuple_10mu_pt_0p5_2.root"
branches = load_root_file(file_path, branches=branches_list, print_branches=True, print_keys=True)

In [ ]:
print(f"Number of events: {branches['event']}")
print(f"branches_list: {branches_list}")

In [ ]:
# Create histograms for all sim_pca_* variables from the raw data, now for muons only (abs(pdgId)=13). 
sim_pca_vars = [
    'sim_pca_pt',
    'sim_pca_eta',
    'sim_pca_lambda',
    'sim_pca_cotTheta',
    'sim_pca_phi',
    'sim_pca_dxy',
    'sim_pca_dz',
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for k, var in enumerate(sim_pca_vars):
    ax = axes[k]
    
    # Flatten all events into a single array
    all_values = []
    for event in range(branches['event']):
        all_values.extend(branches[var][event].flatten())
    
    # only keep values for muons (abs(pdgId) == 13)
    all_values = np.array(all_values)
    all_pdgIds = np.concatenate([branches['sim_pdgId'][event].flatten() for event in range(branches['event'])])
    muon_mask = np.abs(all_pdgIds) == 13
    all_values = all_values[muon_mask]
    vals = np.asarray(all_values)
    vals = vals[np.isfinite(vals)]
    
    # Determine sensible bin boundaries
    if var == 'sim_pca_phi':
        xmin, xmax = -np.pi, np.pi
    else:
        percentile_low, percentile_high = np.percentile(vals, [5, 95])
        xmin, xmax = percentile_low, percentile_high
        if xmin == xmax:
            pad = 1e-6 if xmin == 0 else abs(xmin) * 1e-3
            xmin -= pad
            xmax += pad
    
    n_bins = 80
    in_range = vals[(vals >= xmin) & (vals <= xmax)]
    underflow = np.sum(vals < xmin)
    overflow = np.sum(vals > xmax)
    
    ax.hist(in_range, bins=n_bins, range=(xmin, xmax), alpha=0.85)
    ax.set_title(var)
    ax.set_xlabel(var)
    ax.set_ylabel("Count")
    ax.grid(alpha=0.25)
    
    ax.text(
        0.02, 0.98,
        f"underflow: {underflow}\noverflow: {overflow}\nshown: {len(in_range)}/{len(vals)}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

# Hide the last empty subplot
axes[-1].axis('off')
# title: muons only (abs(pdgId) == 13)
fig.suptitle("Distributions of sim_pca_* variables for muons only (abs(pdgId) == 13)", fontsize=16)

plt.tight_layout()
plt.show()

# print lowest pt value
all_sim_pt = np.concatenate([branches['sim_pt'][event].flatten() for event in range(branches['event'])])
print(f"Lowest sim_pt value: {all_sim_pt.min():.3f} GeV")